# spaCy: fundamentos para analizar texto en español

**Duración estimada:** 35 minutos

## Propósito
En este notebook vas a practicar las operaciones básicas de `spaCy` sobre ejemplos breves. La idea es que puedas mirar una salida, interpretarla y decidir para qué sirve.

## Cómo trabajar este material
- Ejecutá cada bloque y frena a leer la salida antes de avanzar.
- Si usas un asistente de IA, pedile hipótesis o explicaciones alternativas, pero contrastalas siempre con lo que devuelve el código.

## Resultados de aprendizaje
Al finalizar, vas a poder:
- reconocer tokens, lemas y etiquetas gramaticales;
- leer una visualización de dependencias y de entidades;
- identificar grupos nominales;
- comprender, a nivel introductorio, como se agrega una personalización simple al pipeline.


In [1]:
!pip install spacy -q
!python -m spacy download es_core_news_sm -q



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import spacy
from collections import Counter
from spacy import displacy

nlp = spacy.load("es_core_news_sm")


## Caso 1: tokenización, lema y categoría gramatical

En este primer caso vamos a observar cómo `spaCy` divide una oración y que información le asigna a cada token.

**Qué deberías mirar:**
- dónde corta los tokens;
- qué lema propone para cada palabra;
- qué etiquetas POS aparecen con más frecuencia.


In [3]:
doc = nlp("Yo, Matías Barreto, estoy volando hacia Buenos Aires.")
print([token.text for token in doc])


['Yo', ',', 'Matías', 'Barreto', ',', 'estoy', 'volando', 'hacia', 'Buenos', 'Aires', '.']


In [4]:
print("La cantidad de tokens es:", len(doc))


La cantidad de tokens es: 11


In [5]:
print("Tokens del documento")
print("_" * 20)
for token in doc:
    print(token)


Tokens del documento
____________________
Yo
,
Matías
Barreto
,
estoy
volando
hacia
Buenos
Aires
.


In [6]:
for token in doc:
    print(token.text, token.lemma_)


Yo yo
, ,
Matías Matías
Barreto Barreto
, ,
estoy estar
volando volar
hacia hacia
Buenos Buenos
Aires Aires
. .


In [7]:
for token in doc:
    print(token.text, token.pos_, spacy.explain(token.pos_))


Yo PRON pronoun
, PUNCT punctuation
Matías PROPN proper noun
Barreto PROPN proper noun
, PUNCT punctuation
estoy AUX auxiliary
volando VERB verb
hacia ADP adposition
Buenos PROPN proper noun
Aires PROPN proper noun
. PUNCT punctuation


In [8]:
frecuencias = Counter(token.text for token in doc)
print(frecuencias)


Counter({',': 2, 'Yo': 1, 'Matías': 1, 'Barreto': 1, 'estoy': 1, 'volando': 1, 'hacia': 1, 'Buenos': 1, 'Aires': 1, '.': 1})


**Pausa de lectura**

En este caso conviene diferenciar tres niveles:
- el texto original del token (`token.text`);
- la forma base (`token.lemma_`);
- la función gramatical (`token.pos_`).

Esa distinción va a ser clave cuando trabajes con noticias o corpus más grandes.


## Caso 2: lemas y dependencias sintácticas

Ahora vamos a mirar relaciones entre palabras. Las dependencias sirven para ver cómo se organiza una oración y que vínculos gramaticales aparecen.


In [9]:
doc = nlp("En abril de 2026 estaremos realizando una introducción a spaCy para trabajar conceptos básicos de PLN.")
for token in doc:
    print(token.text, token.lemma_)


En en
abril abril
de de
2026 2026
estaremos estarer
realizando realizar
una uno
introducción introducción
a a
spaCy spaCy
para para
trabajar trabajar
conceptos concepto
básicos básico
de de
PLN PLN
. .


In [10]:
from spacy.lang.es.examples import sentences

doc = nlp(sentences[11])
print(doc.text)
for token in doc:
    print(token.text, token.pos_, token.dep_)


¿Cuándo nació José de San Martín?
¿ PUNCT punct
Cuándo PRON obj
nació VERB ROOT
José PROPN nsubj
de ADP case
San PROPN flat
Martín PROPN flat
? PUNCT punct


In [11]:
conteo_pos = Counter(token.pos_ for token in doc)
print(conteo_pos)


Counter({'PROPN': 3, 'PUNCT': 2, 'PRON': 1, 'VERB': 1, 'ADP': 1})


In [12]:
displacy.render(doc, style='dep', jupyter=True, options={'distance': 120})


**Preguntas para guiar tu lectura**

- Cuál parece ser el verbo principal de la oración?
- Qué palabras dependen de ese verbo?
- Qué información te aporta la visualización que no veías tan rápido en la lista de tokens?


El verbo principal de la oración es 'nació', del cual dependen 'Cuándo' como objeto y 'José' como sujeto. La visualización además permite ver que 'José de San Martín' funciona como una unidad gracias a las flechas 'flat' que conectan las tres palabras, algo que en la lista de tokens no era evidente a simple vista

## Caso 3: entidades nombradas

En este caso vamos a mirar cómo `spaCy` detecta nombres propios, lugares y otras entidades relevantes.


In [13]:
text = "Queremos viajar desde Buenos Aires a Mar del Plata y, unos días después, a La Plata."
doc = nlp(text)
displacy.render(doc, style='ent', jupyter=True, options={'distance': 200})


In [14]:
text = "Su nombre es Benjamín; nació en Argentina."
doc = nlp(text)
displacy.render(doc, style='ent', jupyter=True, options={'distance': 200})


In [15]:
displacy.render(doc, style='dep', jupyter=True, options={'distance': 120})


**Importante**

Los modelos no aciertan siempre. Si una entidad te parece discutible, no la des por válida automáticamente: usala como punto de análisis.


## Caso 4: grupos nominales

Los grupos nominales (`noun_chunks`) te ayudan a detectar segmentos como `"mis padres"` o `"Buenos Aires"` cuando funcionan como unidades dentro de la oración.


In [18]:
for token in nlp("Mis padres y amigos viven en Buenos Aires."):
    print(token.text)


Mis
padres
y
amigos
viven
en
Buenos
Aires
.


In [20]:
for chunk in nlp("Mis padres y amigos viven en Buenos Aires.").noun_chunks:
    print(chunk.text)


Mis padres
amigos
Buenos Aires


En este punto conviene comparar la lista completa de tokens con la lista de grupos nominales: no muestran lo mismo ni sirven para la misma tarea.


## Caso 5: extensión opcional - personalizar la lematización

Este caso es más avanzado. No hace falta dominarlo ahora, pero sirve para entender que el pipeline también se puede adaptar a necesidades puntuales.


In [23]:
doc = nlp("Estoy volando hacia Baires")
print([token.text for token in doc])


['Estoy', 'volando', 'hacia', 'Baires']


In [24]:
from spacy.language import Language


In [25]:
@Language.component("custom_lemmatizer")
def custom_lemmatizer(doc):
    for token in doc:
        if token.text == "Baires":
            token.lemma_ = "Buenos Aires"
    return doc

if "custom_lemmatizer" in nlp.pipe_names:
    nlp.remove_pipe("custom_lemmatizer")

nlp.add_pipe("custom_lemmatizer", after="lemmatizer")

doc = nlp("Estoy volando a Baires")
print([token.text for token in doc])
print([token.lemma_ for token in doc])


['Estoy', 'volando', 'a', 'Baires']
['estar', 'volar', 'a', 'Buenos Aires']


## Cierre y continuidad

Antes de pasar al siguiente notebook, comprobá si podés responder estas preguntas sin mirar las celdas anteriores:
- Qué diferencia hay entre token, lema y POS?
- Para qué sirve mirar dependencias?
- Por qué una entidad detectada por el modelo no siempre debe aceptarse sin revisión?

**Actividad breve con IA:** pedile a un asistente que te proponga una oración y que anticipe cómo la tokenizaría `spaCy`. Después corré esa oración en el notebook y compará ambas lecturas.


 - Qué diferencia hay entre token, lema y POS?

Un token es cada unidad mínima de texto que spaCy identifica, como una palabra o signo de puntuación. El lema es la forma base o de diccionario de ese token, por ejemplo 'viajó' a  'viajar'. El POS es la categoría gramatical que le asigna el modelo, como VERB, NOUN o ADJ. Las tres son capas distintas de información sobre la misma palabra.

- Para qué sirve mirar dependencias?

Mirar las dependencias sirve para entender cómo se relacionan las palabras dentro de una oración: quién es el sujeto, cuál es el objeto, qué palabras modifican a cuáles. Esto es útil para tareas más complejas como extracción de información o análisis de sentimiento.

- Por qué una entidad detectada por el modelo no siempre debe aceptarse sin revisión?


Una entidad detectada por el modelo no siempre debe aceptarse sin revisión porque los modelos cometen errores, especialmente con palabras ambiguas, jerga o nombres poco comunes.

**Actividad breve con IA:** pedile a un asistente que te proponga una oración y que anticipe cómo la tokenizaría `spaCy`. Después corré esa oración en el notebook y compará ambas lecturas.


In [26]:
texto3 = "Boca Juniors y River Plate se enfrentan hoy en el estadio Monumental por el Superclásico."

doc3 = nlp(texto3)

print("PALABRA\t\tLEMA\t\tPOS\t\tEXPLICACIÓN")
print("-------\t\t----\t\t---\t\t-----------")
for token in doc3:
    print(f"{token.text:<15}\t{token.lemma_:<15}\t{token.pos_:<10}\t{spacy.explain(token.pos_)}")

PALABRA		LEMA		POS		EXPLICACIÓN
-------		----		---		-----------
Boca           	Boca           	PROPN     	proper noun
Juniors        	Juniors        	PROPN     	proper noun
y              	y              	CCONJ     	coordinating conjunction
River          	River          	PROPN     	proper noun
Plate          	Plate          	PROPN     	proper noun
se             	él             	PRON      	pronoun
enfrentan      	enfrentar      	VERB      	verb
hoy            	hoy            	ADV       	adverb
en             	en             	ADP       	adposition
el             	el             	DET       	determiner
estadio        	estadio        	NOUN      	noun
Monumental     	Monumental     	PROPN     	proper noun
por            	por            	ADP       	adposition
el             	el             	DET       	determiner
Superclásico   	Superclásico   	PROPN     	proper noun
.              	.              	PUNCT     	punctuation
